# Channel against HFSS and the empirical PCA beam

One transmitter comb tooth through the elevation-sweep raster (07-17
20:26–21:28 UTC), plotted on the corrected pointing coordinates
(`az_deg`, `el_deg` from `curation/pointing_table.parquet`). It is compared with
the HFSS polarised beam sampled at the same coordinates, with the residual.

The signal is the tooth minus the mean of the gap channels ±4 away. The model
is `gain × |E_beam* · e_tx|²`. The only fitted parameter is the gain: the
background is removed by the gap subtraction, not fitted, with the beam pointed at the
triangulated transmitter. The mount rolls about the boresight
(`topo = Rx(E)·Rz(A)·body`, as in `eigsep_data.beam_mapping.tx_model`), with the
zero-points and transmitter polarisation taken from the per-frequency fits in
`beam_explorer.ipynb`. Only the gain is fitted in this original baseline.


It is then compared with the empirical PCA-basis beam at $\ell_{\max}=8$
(`derived/beam/empirical_raster_v0003/pca`). The development of that fit
(basis and order selection, optimization, DPSS-vs-PCA and angular-order
comparisons) is in `channel_vs_hfss_debug.ipynb`.

In [ ]:
from pathlib import Path
import json
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import healpy as hp
import pyarrow.parquet as pq

PLOT_BACKEND = os.environ.get('EIGSEP_NOTEBOOK_BACKEND', 'widget')
get_ipython().run_line_magic('matplotlib', PLOT_BACKEND)
os.environ.setdefault('JAX_PLATFORMS', 'cpu')
os.environ.setdefault('JAX_ENABLE_X64', 'true')

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'marjum-2026-07/data').is_dir())
CAMPAIGN = ROOT / 'marjum-2026-07'

import eigsep_data
from eigsep_data import AntennaResolutionPolicy, MetadataIndex
from eigsep_data.beam_sim import read_beam
eigsep_data.set_campaign_root(CAMPAIGN)

RASTER_UTC = ('2026-07-17 20:26:00', '2026-07-17 21:28:40')
FREQ_MHZ = 156.25            # an HFSS slice; every HFSS slice sits on an arm-1 comb tooth
GAP_OFFSET = 4

# Geometry and polarisation from beam_explorer.ipynb (branch B, roll mount).
AZ_SIGN, EL_SIGN = -1.0, -1.0
AZ0, EL0 = 177.0, 0.0        # deg
ALPHA = 45.0                 # transmitter polarisation, deg ccw from East

In [ ]:
t0, t1 = (pd.Timestamp(s, tz='UTC').timestamp() for s in RASTER_UTC)

# --- corrected pointing, joined to the correlator rows on time -------------
pointing = pq.read_table(CAMPAIGN / 'curation/pointing_table.parquet').to_pandas()
pointing = pointing[(pointing.t_utc_s >= t0 - 60) & (pointing.t_utc_s <= t1 + 60)]
policy = AntennaResolutionPolicy.load(CAMPAIGN / 'curation/antenna_resolution.json')
bundle = MetadataIndex(CAMPAIGN / 'data').select(time=(t0, t1)).load_bundle(
    antenna='box-air', missing='skip', resolution_policy=policy)
meta = bundle.meta.reset_index(drop=True).assign(t=lambda x: x.time_best.astype(float))
j = pd.merge_asof(meta.sort_values('t').reset_index(names='bundle_row'),
                  pointing.sort_values('t_utc_s')[['t_utc_s', 'az_deg', 'el_deg', 'quality']],
                  left_on='t', right_on='t_utc_s', direction='nearest', tolerance=0.35)
j = j[(j.quality == 'ok') & (j.rfswitch == 'RFANT')].reset_index(drop=True)

def reversal_glitches(el_deg, tol=15.0):
    """Isolated solved-elevation faults: a sample is good if at least 3 of its
    4 neighbours (+/-1, +/-2) lie within `tol` deg of it across the +/-180
    wrap. The sweeps move ~3 deg per sample, so real data always passes.
    At reversals the IMU readout drops single samples to ~0 or -60 deg while
    the motor is pegged at -180, and those have at most one close neighbour.
    """
    el = np.asarray(el_deg, float)
    close = np.zeros(el.size, int)
    for k in (-2, -1, 1, 2):
        d = np.abs((el - np.roll(el, k) + 180.0) % 360.0 - 180.0)
        edge = slice(0, -k) if k < 0 else slice(-k, None)   # neighbours that wrapped around the array
        d[edge] = 0.0
        close += d <= tol
    return close < 3


glitch = reversal_glitches(j.el_deg.to_numpy())
print('dropped %d elevation glitches' % glitch.sum())
j = j[~glitch].reset_index(drop=True)

T = j.t.to_numpy()
AZ = j.az_deg.to_numpy()
EL = j.el_deg.to_numpy()

# --- the channel -------------------------------------------------------------
BEAM_CART, _, HFSS_FREQS = read_beam()
s = int(np.argmin(np.abs(HFSS_FREQS - FREQ_MHZ)))
ch = int(round(HFSS_FREQS[s] / (250.0 / 1024)))
col = {int(round(f / (250.0 / 1024))): i for i, f in enumerate(bundle.freqs_mhz)}
rows = j.bundle_row.to_numpy()
tooth = bundle.data[rows, col[ch]].astype(float)
gap = 0.5 * (bundle.data[rows, col[ch - GAP_OFFSET]] + bundle.data[rows, col[ch + GAP_OFFSET]])
DATA = tooth - gap
# Whole-band dropouts (e.g. 21:18:00-21:18:35) take the gap down with the tooth.
ok = (gap / np.median(gap) > 1 / 1.5) & (gap / np.median(gap) < 1.5) & np.isfinite(DATA)
print('channel %d = %.4f MHz; %d samples, %d masked as dropouts'
      % (ch, HFSS_FREQS[s], len(DATA), (~ok).sum()))

In [ ]:
# --- HFSS model at the same coordinates -------------------------------------
ant = np.asarray(json.loads((CAMPAIGN / 'curation/horizon_profiles.json').read_text())['antenna_enu_m'])
tx = np.asarray(json.loads((CAMPAIGN / 'curation/transmitter_position.json').read_text())['best_estimate_enu_m'])
tx_dir = (tx - ant) / np.linalg.norm(tx - ant)


def rotation(az_deg, el_deg):
    # topo = Rx(E) @ Rz(A) @ body: azimuth rolls about the boresight, elevation tips.
    A, E = np.deg2rad(az_deg), np.deg2rad(el_deg)
    cA, sA, cE, sE = np.cos(A), np.sin(A), np.cos(E), np.sin(E)
    z, o = np.zeros_like(cA), np.ones_like(cA)
    Rz = np.stack([np.stack([cA, -sA, z], -1), np.stack([sA, cA, z], -1), np.stack([z, z, o], -1)], -2)
    Rx = np.stack([np.stack([o, z, z], -1), np.stack([z, cE, -sE], -1), np.stack([z, sE, cE], -1)], -2)
    return Rx @ Rz


R = rotation(AZ_SIGN * AZ + AZ0, EL_SIGN * EL + EL0)
b = np.einsum('nji,j->ni', R, tx_dir)                       # transmitter direction, body frame
th, ph = np.arccos(np.clip(b[:, 2], -1, 1)), np.mod(np.arctan2(b[:, 1], b[:, 0]), 2 * np.pi)
w = BEAM_CART[s][:, hp.ang2pix(hp.npix2nside(BEAM_CART.shape[2]), th, ph)].T
a = np.deg2rad(ALPHA)
e = np.einsum('nji,j->ni', R, np.array([-np.sin(a), np.cos(a), 0.0]))
e -= np.sum(e * b, 1, keepdims=True) * b
w -= np.sum(w * b, 1, keepdims=True) * b
P = np.abs(np.sum(np.conj(w) * e, 1)) ** 2

# One free parameter: the gain. The background is removed by the gap
# subtraction itself (DATA = tooth - gap); nothing further is fitted for it.
# The gap channel is 1 MHz away with the tones bin-centred and leakage-free, so
# it measures the background under the tooth directly. The published
# v3-beta.1 DPSS model was compared against it on the raster and is
# indistinguishable (mean R^2 0.744 vs 0.745, <=0.003 at every frequency),
# while covering only 13 of the 30 raster files, so the gap is used.
gain = float(np.sum(P[ok] * DATA[ok]) / np.sum(P[ok] ** 2))
MODEL = gain * P
RESID = DATA - MODEL
print('gain %.3g;  fractional RMS %.3f' % (gain, np.linalg.norm(RESID[ok]) / np.linalg.norm(DATA[ok])))

## Models and scores

The PCA beam is the v0003 all-sample fit at $\ell_{\max}=8$: four HFSS-derived spectral modes, each with a freely fitted angular map, jointly fitted with 42 positive tooth gains and a $10^{-4}$ HFSS-centred coefficient prior. Its geometry offsets are fitted too. The HFSS model fits one gain.

Scores are fractional RMS, $\lVert d-m\rVert/\lVert d\rVert$. The all-sample column is **in-sample** for the PCA beam. The held-out column uses the PCA fit trained without the test stripes (12° azimuth stripes), with HFSS gains also fitted on the training stripes only.

In [ ]:
# --- empirical PCA beam, lmax=8 ---------------------------------------------
import hashlib

PRODUCT = CAMPAIGN / 'derived/beam/empirical_raster_v0003'
provenance = json.loads((PRODUCT / 'provenance.json').read_text())
for relative_path, expected in provenance['input_sha256'].items():
    assert hashlib.sha256((ROOT / relative_path).read_bytes()).hexdigest() == expected, relative_path
for relative_path, expected in provenance['artifact_sha256'].items():
    assert hashlib.sha256((PRODUCT / relative_path).read_bytes()).hexdigest() == expected, relative_path
assert provenance['lmax'] == 8
with np.load(PRODUCT / 'pca/diagnostics.npz') as z:
    pca = {key: z[key] for key in z.files}
assert np.allclose(T, pca['t'], rtol=0, atol=1e-5)
# The empirical fit excludes the opening near-zero-elevation azimuth slew
# (56 samples, 20:26:00-20:26:30); both models are shown on the same samples.
FIT_OK = ok & ~pca['el0_slew'].astype(bool)
fidx = int(np.flatnonzero(pca['channels'] == ch)[0])
assert np.array_equal(FIT_OK, pca['good'][fidx])

def saved_beam_model(product):
    # Evaluate a saved beam through read_beam at the corrected coordinates, with
    # its jointly fitted geometry offsets and gain (no refit here).
    with np.load(product / 'pca/diagnostics.npz') as z:
        diag = {key: z[key] for key in z.files}
    assert np.allclose(T, diag['t'], rtol=0, atol=1e-5)
    beam, _, frequencies = read_beam(product / 'pca/empirical_beam.npz')
    bidx = int(np.argmin(abs(frequencies - HFSS_FREQS[s])))
    assert np.isclose(frequencies[bidx], HFSS_FREQS[s])
    pa, paz, pel, peast, pnorth = diag['params']
    direction = tx + np.array([peast, pnorth, 0.]) - ant
    direction /= np.linalg.norm(direction)
    rot = rotation(-AZ + AZ0 + paz, -EL + EL0 + pel)
    body = np.einsum('nji,j->ni', rot, direction)
    theta = np.arccos(np.clip(body[:, 2], -1, 1))
    phi = np.mod(np.arctan2(body[:, 1], body[:, 0]), 2 * np.pi)
    pixel, weight = hp.get_interp_weights(hp.npix2nside(beam.shape[-1]), theta, phi)
    field = sum(beam[bidx][:, pixel[k]] * weight[k] for k in range(4)).T
    alpha = np.deg2rad(ALPHA + pa + 90 * ((ch // 8) % 2))
    electric = np.einsum('nji,j->ni', rot, [-np.sin(alpha), np.cos(alpha), 0.])
    electric -= np.sum(electric * body, axis=1)[:, None] * body
    model = diag['exported_gain'][fidx] * abs(np.sum(field.conj() * electric, axis=1)) ** 2
    assert np.allclose(model, diag['empirical_exported'][fidx], rtol=1e-9, atol=1e-5)
    return model, diag


PCA_MODEL, _ = saved_beam_model(PRODUCT)
PCA_RESID = DATA - PCA_MODEL

test = pca['split'] == 2
def frac_rms(resid, use):
    return np.linalg.norm(resid[use]) / np.linalg.norm(DATA[use])
display(pd.DataFrame([
    dict(model='HFSS (gain only)', all_samples=frac_rms(RESID, FIT_OK),
         test_stripes_heldout=frac_rms(DATA - pca['hfss'][fidx], FIT_OK & test)),
    dict(model='PCA lmax=8', all_samples=frac_rms(PCA_RESID, FIT_OK),
         test_stripes_heldout=frac_rms(DATA - pca['empirical_heldout'][fidx], FIT_OK & test)),
]).set_index('model').round(4))

In [ ]:
plt.rcParams.update({'font.size': 14, 'axes.spines.top': False, 'axes.spines.right': False})
az_plot = (AZ + 180.0) % 360.0 - 180.0          # the raster crosses 0/360; keep it contiguous
time = pd.to_datetime(T, unit='s', utc=True)

# Shared color scales for both models: log data/model, symmetric residual set by HFSS.
lo, hi = np.percentile(DATA[FIT_OK & (DATA > 0)], [1, 99.5])
norm = LogNorm(vmin=lo, vmax=hi)
rlim = np.percentile(np.abs(RESID[FIT_OK]), 98)


def plot_maps(model, name):
    fig, axes = plt.subplots(1, 3, figsize=(19, 7), sharex=True, sharey=True,
                             constrained_layout=True)
    panels = [('Data (tooth − gap)', DATA, dict(cmap='viridis', norm=norm)),
              (name + ' model', model, dict(cmap='viridis', norm=norm)),
              ('Residual (data − model)', DATA - model, dict(cmap='RdBu_r', vmin=-rlim, vmax=rlim))]
    for ax, (title, value, kw) in zip(axes, panels):
        sc = ax.scatter(az_plot[FIT_OK], EL[FIT_OK], c=value[FIT_OK], s=4, **kw)
        ax.set_title(title)
        ax.set_xlabel('Azimuth [deg]')
        fig.colorbar(sc, ax=ax, orientation='horizontal', pad=0.1, shrink=0.85, label='Counts')
    axes[0].set_ylabel('Elevation [deg]')
    fig.suptitle('%s, %.3f MHz, raster %s–%s UTC'
                 % (name, HFSS_FREQS[s], RASTER_UTC[0][-8:], RASTER_UTC[1][-8:]), fontsize=17)
    plt.show()


def plot_time(model, name):
    fig, ax = plt.subplots(figsize=(16, 6.5), constrained_layout=True)
    ax.plot(time[FIT_OK], DATA[FIT_OK], '.', ms=2, color='0.25', label='data')
    ax.plot(time[FIT_OK], model[FIT_OK], '.', ms=2, color='tab:red', label=name + ' model')
    ax.plot(time[FIT_OK], (DATA - model)[FIT_OK], '.', ms=2, color='tab:blue', label='residual')
    ax.axhline(0, color='0.6', lw=1)
    ax.set_ylabel('Counts')
    ax.set_xlabel('Time [UTC]')
    ax.set_title('%s, %.3f MHz through the raster' % (name, HFSS_FREQS[s]))
    ax.legend(loc='upper right', markerscale=6)
    plt.show()

## HFSS beam

Data, model and residual on the corrected az/el, then against time. Both models use the same samples (the opening el≈0 azimuth slew is excluded) and the same color scales.

In [ ]:
plot_maps(MODEL, 'HFSS')

In [ ]:
plot_time(MODEL, 'HFSS')

## Empirical PCA beam, $\ell_{\max}=8$

In [ ]:
plot_maps(PCA_MODEL, 'PCA ℓmax=8')

In [ ]:
plot_time(PCA_MODEL, 'PCA ℓmax=8')

## Masking the sweep-reversal corner (az 0–25°, |el| > 165°)

The PCA residual map shows a coherent corner at az 0–25°, |el| > 165°: 76 sweep-reversal samples, 21:13–21:19 UTC. There, one transmitter arm reads about 25% below both models and the other about 7% above. The test is whether these samples skew the fit. `derived/beam/empirical_raster_v0005` repeats the v0003 PCA $\ell_{\max}=8$ fit with this corner removed from fitting in every channel. Everything else is identical: HFSS-prior initialization, $10^{-4}$ penalty, geometry, 42 teeth, and splits. It was generated by `data-analysis/scripts/marjum-2026-07/empirical_beam_pca_polemask.py`. Both fits converged.

Scores exclude the corner, so both fits are compared on the same samples. The corner column shows how each model predicts the masked samples. The corner has no test-stripe samples.

In [ ]:
MASKED_PRODUCT = CAMPAIGN / 'derived/beam/empirical_raster_v0005'
masked_provenance = json.loads((MASKED_PRODUCT / 'provenance.json').read_text())
for relative_path, expected in masked_provenance['input_sha256'].items():
    assert hashlib.sha256((ROOT / relative_path).read_bytes()).hexdigest() == expected, relative_path
for relative_path, expected in masked_provenance['artifact_sha256'].items():
    assert hashlib.sha256((MASKED_PRODUCT / relative_path).read_bytes()).hexdigest() == expected, relative_path
MASKED_MODEL, masked = saved_beam_model(MASKED_PRODUCT)
CORNER = masked['corner'].astype(bool)
assert CORNER.sum() == 76 and not masked['good'][fidx][CORNER].any()
assert np.array_equal(FIT_OK & ~CORNER, masked['good'][fidx])

validation = pca['split'] == 1
rows = []
for name, model, heldout in [('HFSS (gain only)', MODEL, pca['hfss'][fidx]),
                             ('PCA lmax=8', PCA_MODEL, pca['empirical_heldout'][fidx]),
                             ('PCA lmax=8, corner masked', MASKED_MODEL, masked['empirical_heldout'][fidx])]:
    rows.append(dict(model=name,
                     outside_corner=frac_rms(DATA - model, FIT_OK & ~CORNER),
                     corner=frac_rms(DATA - model, FIT_OK & CORNER),
                     validation_heldout=frac_rms(DATA - heldout, FIT_OK & ~CORNER & validation),
                     test_heldout=frac_rms(DATA - heldout, FIT_OK & ~CORNER & test)))
display(pd.DataFrame(rows).set_index('model').round(4))

In [ ]:
plot_maps(MASKED_MODEL, 'PCA ℓmax=8, corner masked')

In [ ]:
plot_time(MASKED_MODEL, 'PCA ℓmax=8, corner masked')

## Caveats

- The PCA beam is exploratory, not a released product. Its all-sample score is in-sample; the held-out column is the fairer comparison. The test stripes are inherited from earlier exploration, so they are not a fresh blind test.
- Power-only measurements leave the absolute phase and the circular-polarization response unconstrained. Directions the raster never visited stay close to the HFSS prior.
- The campaign INDEX labels this raster's every-eighth-channel signal as the instrument self-comb. Following Aaron and `beam_explorer`, it is treated here as the transmitter; that archive conflict is unresolved.
- The corner-masked fit (v0005) changes one masking choice only; it is a sensitivity test, not a replacement beam.